<div class="alert alert-block alert-success" style="font-family: Times New Roman">
    <h4><strong>Laboratory Activity 6</strong></h4>

<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
    <b>Instruction:</b> Convert the following CNN architecture diagram into a PyTorch CNN Architecture.
</p>

<center><img src="quickdraw.png" width="400px"></center>
</div>

In [17]:
import torch
import torch.nn as nn

"""
IMPORTS AND MODEL INITIALIZATION
This initial setup imports the necessary PyTorch libraries and defines the CustomCNN class. 
By inheriting from nn.Module, the custom class is integrated with PyTorch's backend, 
gaining built-in parameter tracking and gradient computation functionalities[cite: 5]. 
The __init__ constructor sets up the network to default to 10 classes, aligning with the 
10 categorical digits (0-9) of the MNIST dataset used in this laboratory task[cite: 5].
"""
class CustomCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CustomCNN, self).__init__()
        
        """
        FIRST CONVOLUTIONAL BLOCK
        This block is the network's first feature extractor. The nn.Conv2d layer takes a 
        single-channel image (like the 28x28 grayscale MNIST images[cite: 5]) and outputs 
        32 feature maps using 3x3 filters. The padding=1 argument ensures the image's 
        spatial dimensions are preserved during the convolution operation. Next, the ReLU 
        activation introduces non-linearity, and the MaxPool2d layer downsamples the 
        spatial dimensions by a factor of 2, retaining only the most prominent features 
        while reducing the computational load.
        """
        # Input shape: (1, 28, 28)
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.relu1 = nn.ReLU()
        self.maxpool1 = nn.MaxPool2d(kernel_size=2, stride=2, padding=1)
        
        """
        SECOND CONVOLUTIONAL BLOCK
        This deeper sequence of convolutions progressively increases the channel depth from 
        32 to 64, then 128, and finally 256. Stacking multiple convolutional layers with 
        ReLU activations before pooling allows the network to learn increasingly complex, 
        high-level hierarchical features. The final MaxPool2d compresses these thick 
        feature maps one last time before they are sent to the classification head.
        """
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.relu3 = nn.ReLU()
        
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1)
        self.relu4 = nn.ReLU()
        
        self.maxpool2 = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        
        """
        DROPOUT, FLATTEN, AND FULLY CONNECTED NETWORK
        This section transitions the model from spatial feature maps to a flat classification 
        vector. nn.Dropout(p=0.2) is a regularization layer that randomly ignores 20% of the 
        neurons during training, preventing the model from overfitting to the training data. 
        nn.Flatten() unrolls the 3D tensor grid into a 1D vector of 12,544 elements. The Fully 
        Connected (nn.Linear) layers then map these 12,544 features down to 1000, then 500, 
        and ultimately 10 target classes[cite: 5]. The Softmax function converts the final 
        raw output into a usable probability distribution between 0.0 and 1.0.
        """
        self.dropout = nn.Dropout(p=0.2)
        self.flatten = nn.Flatten()
        
        # The calculated input dimension is 12544 (256 channels * 7 height * 7 width)
        self.fcn1 = nn.Linear(in_features=12544, out_features=1000)
        self.relu_fcn1 = nn.ReLU()
        
        self.fcn2 = nn.Linear(in_features=1000, out_features=500)
        self.relu_fcn2 = nn.ReLU()
        
        # The final output corresponds to the number of classification classes
        self.fcn3 = nn.Linear(in_features=500, out_features=num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        """
        FORWARD PASS EXECUTION
        The forward method explicitly dictates the step-by-step data flow through the network 
       [cite: 5]. When an image tensor 'x' is fed to the model, it passes sequentially through 
        the convolution blocks, flattening operations, and the dense layers as defined here. 
        PyTorch dynamically tracks this data flow to build a computational graph, which is 
        required later to calculate gradients during backward propagation.
        """
        # Block 1
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.maxpool1(x)
        
        # Block 2
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.conv3(x)
        x = self.relu3(x)
        x = self.conv4(x)
        x = self.relu4(x)
        x = self.maxpool2(x)
        
        # Dropout & Flatten
        x = self.dropout(x)
        x = self.flatten(x)
        
        # FCN
        x = self.fcn1(x)
        x = self.relu_fcn1(x)
        x = self.fcn2(x)
        x = self.relu_fcn2(x)
        x = self.fcn3(x)
        x = self.softmax(x)
        
        return x

"""
MODEL INSTANTIATION
This block brings the defined class to life by instantiating it as an object named 'model', 
explicitly passing 10 as the number of output classes. The print(model) command outputs a 
summary of the network's blueprint, verifying that all layers were successfully initialized 
and registered within the PyTorch architecture.
"""
# Example instantiation 
model = CustomCNN(num_classes=10)
print(model)

CustomCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu1): ReLU()
  (maxpool1): MaxPool2d(kernel_size=2, stride=2, padding=1, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu2): ReLU()
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu3): ReLU()
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu4): ReLU()
  (maxpool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.2, inplace=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fcn1): Linear(in_features=12544, out_features=1000, bias=True)
  (relu_fcn1): ReLU()
  (fcn2): Linear(in_features=1000, out_features=500, bias=True)
  (relu_fcn2): ReLU()
  (fcn3): Linear(in_features=500, out_features=10, bias=True)
  (softmax): Softmax(dim=1)
)


**Hierarchical Feature Expansion (Conv Blocks 1 & 2)**
The output reveals a classic deep learning design philosophy: increasing channel depth while reducing spatial dimensions.

The network starts at conv1 with a single input channel (suitable for grayscale images) and expands it to 32 channels.

As the data flows through conv2, conv3, and conv4, the model aggressively scales the feature maps from 32 to 64, 128, and finally 256 channels. This indicates that as the network goes deeper, it shifts from analyzing simple, low-level features (like lines and edges) to highly complex, abstract patterns.

The two MaxPool2d layers act as spatial bottlenecks, condensing the physical grid size of the image to keep the mathematical operations efficient and to make the network less sensitive to minor shifts or rotations in the input image.

**Dimensional Transition (Dropout & Flatten)**
The (dropout) and (flatten) layers represent the bridge between the spatial, "vision" part of the network and the mathematical classification part. The Flatten layer explicitly shows start_dim=1 and end_dim=-1, meaning it preserves the batch size (dimension 0) but crushes the remaining 3D tensor (channels, height, width) into a single 1D array. The Dropout layer confirms its probability is set to p=0.2, verifying that 20% of the network's signals will be randomly dropped during training to prevent memorization (overfitting).

**Classification Funnel (FCN 1-3 & Softmax)**
The final linear layers show a rapid, deliberate funneling of data.

fcn1 intakes the massive 12544 element vector produced by the flatten layer and compresses it into a 1000-node dense layer.

fcn2 further compresses this to 500 nodes.

fcn3 outputs exactly 10 features, confirming the network is perfectly tailored for a 10-class classification problem (such as identifying the digits 0-9).

Finally, the Softmax(dim=1) layer confirms that these 10 raw scores will be mathematically squashed into a clean probability distribution that adds up to 100%, allowing the model to make a definitive final prediction.